# 19.4 Capstone — Concurrent API Aggregator

**Prerequisites:** 11 Socket Programming, 12 Concurrency (**12.4**, **12.5**), 15 Testing (**15.4**, **15.5**), 16 Type Hints, 18 Working with APIs (all of it)  
**Target:** Python 3.12+ (`pydantic` and `httpx` installed, as in **18.4**/**18.5**)

### What you'll build
A **status-dashboard aggregator** — the thing every ops team builds eventually: poll the
`/health` endpoint of 12 microservices, validate every payload at the trust boundary, and
aggregate the results into **one fleet report** — fast, and without trusting anyone.

- **The fake fleet** — 12 local services with deterministic bad behaviour (no network, no secrets)
- **A client in three layers** — `Session` + timeout always, retries on an adapter, `pydantic` at the boundary
- **Three pollers, measured** — sequential, `ThreadPoolExecutor`, `asyncio` + `httpx`
- **One report** — healthy/degraded/unreachable/invalid, worst latency, version spread
- **Tests** — a `FakeSession` for units, a real local server fixture for integration

---

## The problem

You run a platform of twelve microservices. Each exposes `GET /<service>/health` and is
*supposed* to answer with:

```json
{"status": "healthy", "latency_ms": 50.0, "version": "2.3.1"}
```

The on-call engineer needs one consolidated answer — *what is the fleet doing right now?* —
and reality is not cooperating:

- most services are fine, but their latencies differ by an order of magnitude
- **payments** returns `503` intermittently (a deploy is mid-flight)
- **search-legacy** is simply down
- **metrics** sits behind a broken proxy that truncates its body mid-JSON
- **reports** shipped a redesign yesterday and renamed every field

The job: **one report, fast, without trusting anyone.** A poll that crashes because one
service lied is worse than no poll at all.

### What this capstone stitches together

| Piece of the build | Where you learned it |
|---|---|
| A real local HTTP fleet — offline, no secrets | the fake-API pattern of **18.1–18.5** |
| `Session`, connection pooling, a timeout on *every* request | **11.5**, **18.5** |
| `urllib3.Retry` mounted on an `HTTPAdapter`, `GET` only | **18.3** |
| A `pydantic` contract at the trust boundary | **18.4** |
| `ThreadPoolExecutor` + `as_completed` | **12.4** |
| `asyncio.gather` + `Semaphore`, via `httpx` | **12.5**, **18.5** |
| `FakeSession` unit tests + a real-server integration test | **15.4**, **15.5**, **18.5** |
| The aligned report table, with format specs | **1.3** |

## Part 1 — The fake fleet

Twelve "services" served by one `ThreadingHTTPServer` inside this process — the same
pattern every notebook in folder 18 used. Real HTTP over a real socket, zero network,
and every quirk is **deterministic**: the flakiness is scripted, not random.

In [ ]:
# ---- The fleet: 12 fake microservices in one local HTTP server ----
import http.server
import json
import socket
import threading
import time
import urllib.parse

FLEET: dict[str, dict] = {
    # name              artificial latency + what /health actually does
    "gateway":       {"latency": 0.05, "status": "healthy",  "version": "2.3.1"},
    "auth":          {"latency": 0.08, "status": "healthy",  "version": "2.3.1"},
    "users":         {"latency": 0.10, "status": "healthy",  "version": "2.3.0"},
    "catalog":       {"latency": 0.12, "status": "healthy",  "version": "2.3.1"},
    "orders":        {"latency": 0.15, "status": "healthy",  "version": "2.2.9"},
    "inventory":     {"latency": 0.18, "status": "healthy",  "version": "2.3.0"},
    "shipping":      {"latency": 0.20, "status": "degraded", "version": "2.2.9"},
    "notifications": {"latency": 0.40, "status": "healthy",  "version": "2.3.1"},  # the slow one
    "payments":      {"latency": 0.10, "status": "healthy",  "version": "2.3.1",
                      "flaky": True},       # 503 on its FIRST hit, then recovers
    "search-legacy": {"latency": 0.05, "down": True},        # always 503
    "metrics":       {"latency": 0.09, "malformed": True},   # 200, but the body is not JSON
    "reports":       {"latency": 0.07, "drifted": True},     # 200, valid JSON, wrong schema
}

HITS: dict[str, int] = {name: 0 for name in FLEET}
HITS_LOCK = threading.Lock()          # handler threads run concurrently (12.2)


class FleetHandler(http.server.BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def log_message(self, *args):
        """Silence the default logging."""

    def _send(self, status: int, body: bytes) -> None:
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        path = urllib.parse.urlparse(self.path).path         # /<service>/health
        parts = path.strip("/").split("/")
        if len(parts) != 2 or parts[1] != "health" or parts[0] not in FLEET:
            return self._send(404, b'{"error": "no such service"}')

        name, spec = parts[0], FLEET[parts[0]]
        with HITS_LOCK:
            HITS[name] += 1
            hit = HITS[name]
        time.sleep(spec["latency"])                          # deterministic artificial latency

        if spec.get("down") or (spec.get("flaky") and hit == 1):
            return self._send(503, b'{"error": "service unavailable"}')
        if spec.get("malformed"):                            # a "proxy" truncated the body
            return self._send(200, b'{"status": "healthy", "latency_ms": ')
        if spec.get("drifted"):                              # yesterday's "redesign"
            return self._send(200, json.dumps(
                {"state": "up", "latencyMs": "70ms", "ver": 4}).encode("utf-8"))
        payload = {"status": spec["status"],
                   "latency_ms": spec["latency"] * 1000,
                   "version": spec["version"]}
        return self._send(200, json.dumps(payload).encode("utf-8"))


class QuietServer(http.server.ThreadingHTTPServer):
    daemon_threads = True

    def handle_error(self, *args):
        """A client hanging up is normal."""


def start_fleet() -> tuple[QuietServer, str]:
    probe = socket.socket()
    probe.bind(("127.0.0.1", 0))
    port = probe.getsockname()[1]
    probe.close()
    server = QuietServer(("127.0.0.1", port), FleetHandler)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{port}"


SERVER, BASE = start_fleet()

print(f"fleet of {len(FLEET)} services on {BASE}\n")
print(f"{'service':<15}{'latency':>9}  behaviour")
print("-" * 58)
for name, spec in FLEET.items():
    quirk = ("always 503" if spec.get("down")
             else "503 on first hit, then fine" if spec.get("flaky")
             else "returns broken JSON" if spec.get("malformed")
             else "schema drifted" if spec.get("drifted")
             else f"{spec['status']}, v{spec['version']}")
    print(f"{name:<15}{spec['latency'] * 1000:>7.0f}ms  {quirk}")

In [ ]:
# A first look at what the fleet actually sends — before building anything.
import requests


def peek_at_fleet() -> None:
    """Plain requests, no machinery yet. A function, so no Response object
    outlives it — a kept response holds its socket open (see the teardown)."""
    with requests.Session() as peek:
        print("gateway sends :", peek.get(f"{BASE}/gateway/health", timeout=5).json())

        suspect = peek.get(f"{BASE}/metrics/health", timeout=5)
        print(f"metrics sends : {suspect.text!r}")
        print(f"                status {suspect.status_code},"
              f" Content-Type {suspect.headers['Content-Type']!r}  <- both lies")

        print("reports sends :", peek.get(f"{BASE}/reports/health", timeout=5).json())


peek_at_fleet()

Look at what `metrics` did: **`200 OK`, `Content-Type: application/json` — and a body
that is not JSON.** A status code is a claim, not a guarantee. And `reports` returns
perfectly valid JSON in a shape your code has never seen.

🔴 This is **18.4**'s trust boundary in miniature: everything arriving from an API is
untrusted input — not because the providers are hostile, but because you do not control
them. The client we build next assumes exactly nothing.

## Part 2 — The client, layer by layer

### Layer 1: a `Session`, a timeout, and retries

Three decisions before a single request is made:

1. **One `Session` for the whole poll.** Pooled connections — the cheapest win there is
   (**18.5** measured it).
2. 🔴 **A timeout on every request, no exceptions.** Under concurrency a request with no
   timeout does not just hang — it **holds a worker forever**, and a pool can be fully
   consumed by hung requests (**18.5**). We set `(connect, read)` separately (**18.1**).
3. **Retries on the adapter, `GET` only.** `/health` is a `GET`, `GET` is idempotent
   (**18.1**), so retrying is safe — `urllib3.Retry` does it below the surface (**18.3**),
   with backoff and jitter so a struggling service is not hammered.

In [ ]:
from requests.adapters import HTTPAdapter
from urllib3.util import Retry

TIMEOUT: tuple[float, float] = (3.05, 2.0)        # (connect, read) — 18.1, 18.5


def build_session() -> requests.Session:
    """One Session for the whole poll: pooled connections + automatic retries."""
    policy = Retry(
        total=2,                                  # the request + at most 2 retries
        backoff_factor=0.1,
        backoff_jitter=0.05,                      # 18.3: jitter, so clients never synchronise
        status_forcelist=[502, 503, 504],         # transient server trouble only
        allowed_methods={"GET"},                  # GET is idempotent - safe to repeat (18.1)
        raise_on_status=False,
    )
    session = requests.Session()
    adapter = HTTPAdapter(max_retries=policy, pool_maxsize=len(FLEET))
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session


SESSION = build_session()
print("session ready:")
print(f"   timeout     : connect {TIMEOUT[0]}s, read {TIMEOUT[1]}s - on EVERY request")
print("   retry policy: up to 2 retries, 502/503/504 only, GET only, jittered backoff")
print(f"   pool size   : {len(FLEET)} connections - one per service, reused all notebook")

### Layer 2: the contract, enforced with `pydantic`

`response.json()` returns `Any`, and **16.1** showed how far `Any` spreads. So the payload
meets a `pydantic` model (**18.4**) *at the boundary*, before any aggregation logic sees it:

- `status` must be one of the two words the dashboard understands — `Literal`
- `latency_ms` must be a non-negative number (lax mode will accept `"50"` and coerce —
  a feature at a JSON boundary)
- `version` must look like a semantic version, enforced with a regex `pattern` (**09**)

In [ ]:
from typing import Annotated, Literal

from pydantic import BaseModel, Field, ValidationError


class HealthReport(BaseModel):
    """The contract: what a well-behaved /health payload must look like."""
    status: Literal["healthy", "degraded"]
    latency_ms: Annotated[float, Field(ge=0)]
    version: Annotated[str, Field(pattern=r"^\d+\.\d+\.\d+$")]


good = HealthReport.model_validate(
    {"status": "healthy", "latency_ms": 50.0, "version": "2.3.1"})
print("a valid payload   ->", good)

print("\nthe reports payload against the same contract:")
try:
    HealthReport.model_validate({"state": "up", "latencyMs": "70ms", "ver": 4})
except ValidationError as exc:
    print(f"   ValidationError: {exc.error_count()} problems, all reported at once (18.4)")
    for error in exc.errors():
        location = ".".join(str(part) for part in error["loc"])
        print(f"      {location:<12} {error['msg']}")

### Layer 3: the probe — every failure becomes data

One function does one poll of one service. 🔴 **It never raises.** A poll of twelve
services must not die because one of them lied — the **18.4** page-parsing lesson
(best-effort, and *log* the rejects) applied to a fleet. Every way a service can
disappoint is mapped to a structured verdict:

| What happened | Caught as | Verdict |
|---|---|---|
| timeout, connection refused, `5xx` even after retries | `requests.RequestException` / `HTTPError` | **UNREACHABLE** |
| `200`, but the body is not JSON | `requests.JSONDecodeError` | **INVALID** |
| JSON, but it breaks the contract | `pydantic.ValidationError` | **INVALID** |
| valid payload saying `"degraded"` | — | **DEGRADED** |
| valid payload saying `"healthy"` | — | **HEALTHY** |

🔴 `requests.JSONDecodeError` **is a subclass of** `RequestException`, so its `except`
clause must come first — the interpreter takes the first matching clause (**06**).

🔴 Two hygiene details that earn their keep. The response is used as a **context
manager**, so it is closed on every path, including the `raise_for_status` one. And the
function returns plain `ProbeResult` **data** — never the response itself. A
`requests.Response` you keep alive (a notebook global, a cache entry, a stored
exception) quietly holds its socket's file descriptor open *even after the session is
closed*; that is why the first-look cell above was a function that extracted `.json()`
and `.text` instead of keeping responses around. Hold data, not responses — the
teardown audit at the end of this notebook exposes the leak as a leftover server
thread if you break the rule.

In [ ]:
import enum
from dataclasses import dataclass


class Outcome(enum.StrEnum):                      # StrEnum: 3.11+, formats as its value
    HEALTHY = "healthy"
    DEGRADED = "degraded"
    UNREACHABLE = "unreachable"
    INVALID = "invalid"


@dataclass(frozen=True, slots=True)
class ProbeResult:
    """One service, one poll: the verdict plus everything the report needs."""
    service: str
    outcome: Outcome
    report: HealthReport | None                   # None unless the payload validated
    detail: str
    elapsed_ms: float                             # measured by US, not self-reported


def probe_service(session: requests.Session, base: str, name: str,
                  timeout: tuple[float, float] = TIMEOUT) -> ProbeResult:
    """Poll one /health endpoint. Never raises - every failure becomes a verdict."""
    started = time.perf_counter()

    def done(outcome: Outcome, report: HealthReport | None = None,
             detail: str = "") -> ProbeResult:
        return ProbeResult(name, outcome, report, detail,
                           (time.perf_counter() - started) * 1000)

    try:
        with session.get(f"{base}/{name}/health", timeout=timeout) as response:
            response.raise_for_status()           # `with` closes even on this raise
            payload = response.json()
    except requests.JSONDecodeError:              # before RequestException - it IS one
        return done(Outcome.INVALID, detail="body is not JSON")
    except requests.HTTPError as exc:
        return done(Outcome.UNREACHABLE, detail=f"HTTP {exc.response.status_code}")
    except requests.RequestException as exc:
        return done(Outcome.UNREACHABLE, detail=type(exc).__name__)

    try:
        report = HealthReport.model_validate(payload)
    except ValidationError as exc:
        return done(Outcome.INVALID, detail=f"{exc.error_count()} schema error(s)")

    outcome = Outcome.DEGRADED if report.status == "degraded" else Outcome.HEALTHY
    return done(outcome, report=report)


print(probe_service(SESSION, BASE, "gateway"))

In [ ]:
print("--- payments: flaky, and this is its first hit ever ---")
before = HITS["payments"]
result = probe_service(SESSION, BASE, "payments")
print(f"   caller saw : {result.outcome.value} in {result.elapsed_ms:.0f} ms")
print(f"   server saw : {HITS['payments'] - before} requests"
      "   <- urllib3 retried the 503 for us (18.3)")

print("\n--- the services that cannot be trusted ---")
for name in ("metrics", "reports", "search-legacy"):
    result = probe_service(SESSION, BASE, name)
    print(f"   {name:<14} {result.outcome.value:<12} {result.elapsed_ms:>5.0f} ms  {result.detail}")

Read the `payments` lines again: **the caller made one call and saw one healthy result;
the server saw two requests.** The `503` was absorbed by the adapter's retry policy —
resilience the aggregation code never has to know about. (Re-run the cell and the server
sees one request: the flaky deploy has "finished", so there is nothing left to retry.)

And the three liars produced **data, not exceptions**: `metrics` and `reports` are
`invalid` with a reason, `search-legacy` is `unreachable` — after costing noticeably
more wall time than its 50 ms latency, because the adapter retried it twice with backoff
before giving up. 🔴 Retries are not free; that cost returns in the next section.

## Part 3 — Sequential baseline, measured

The obvious implementation: probe the twelve services one after another. Before running
it, do the arithmetic the way **12.4** taught: each probe is almost pure *waiting*, so
the total should be roughly the **sum** of the per-service times.

In [ ]:
def poll_fleet_sequential() -> list[ProbeResult]:
    return [probe_service(SESSION, BASE, name) for name in FLEET]


started = time.perf_counter()
sequential = poll_fleet_sequential()
sequential_s = time.perf_counter() - started

for result in sequential:
    print(f"   {result.service:<15}{result.outcome.value:<13}{result.elapsed_ms:>7.0f} ms")

configured = sum(spec["latency"] for spec in FLEET.values())
print("-" * 42)
print(f"   {'sum of configured latencies':<28}{configured * 1000:>7.0f} ms")
print(f"   {'measured wall clock':<28}{sequential_s * 1000:>7.0f} ms")

The measured total tracks the sum of the configured latencies, plus one visible outlier:
**`search-legacy` costs several times its 50 ms latency**, because every poll pays for
three attempts plus backoff sleeps before declaring it unreachable. The exact totals
shift a little between runs — loopback and `time.sleep` are only so precise — but the
*shape* is fixed: **sequential cost = the sum of every wait, including the waits you
spend retrying a dead service.**

Twelve services is about two seconds. At 100 services this poll takes a quarter of a
minute — for a *status dashboard*. And **17.5**/**18.5** told us a profiler would show
nothing: the time is spent blocked in socket reads, not in code. The fix is not faster
code; it is overlapping the waiting (**12.4**).

## Part 4 — Threads: `ThreadPoolExecutor`, measured

A thread waiting on a socket has released the GIL (**12.1**), so I/O waits overlap
almost perfectly. `as_completed` (**12.4**) hands back each result the moment it
arrives — a dashboard can render the fast services while the slow ones are still
in flight.

One `Session` is shared across the workers: `requests` pools per-host connections
internally, and `pool_maxsize=len(FLEET)` above sized it for exactly this moment.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def poll_fleet_threaded(max_workers: int = len(FLEET)) -> tuple[list[ProbeResult], list[str]]:
    arrival: list[str] = []
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = [pool.submit(probe_service, SESSION, BASE, name) for name in FLEET]
        results = []
        for future in as_completed(futures):      # yields in COMPLETION order (12.4)
            result = future.result()
            arrival.append(result.service)
            results.append(result)
    return results, arrival


started = time.perf_counter()
threaded, arrival = poll_fleet_threaded()
threaded_s = time.perf_counter() - started

print("completion order this run (first back first):")
print("   " + " -> ".join(arrival[:6]))
print("   " + " -> ".join(arrival[6:]))
print()
print(f"   {'sequential':<12}{sequential_s * 1000:>7.0f} ms")
print(f"   {'threads':<12}{threaded_s * 1000:>7.0f} ms"
      f"   ({sequential_s / threaded_s:.1f}x faster this run)")

Twelve waits overlapped: the poll now costs roughly its **slowest member** — the greater
of `notifications` (400 ms) and `search-legacy`-with-retries — instead of the sum of all
twelve. Rebuilds of this notebook have printed anywhere from roughly 3x to 5x for that
ratio; whatever your run says, it is the collapse in *shape* that matters.

🔴 **Do not read too much into the exact ordering or ratio.** On loopback, services
whose latencies differ by a few milliseconds finish in an order that changes from run to
run — **18.5** hit exactly this instability and concluded that the lesson is never a
specific number. What *is* stable, and worth taking away:

1. the fast services surface first and the slow or retried ones last;
2. total time collapses from *sum of waits* to roughly *max of waits*;
3. `max_workers` is the knob that bounds how hard you hit the fleet at once (**18.3**).

## Part 5 — asyncio with `httpx`, measured

`requests` is synchronous and cannot be awaited, so the async poller uses `httpx`
(**18.5**). Two deliberate choices:

- **`asyncio.Semaphore(4)`** caps requests in flight (**12.5**, **18.5**) — twelve at
  once is fine for our own fleet, but a semaphore is how you stay polite against
  services that rate-limit (**18.3**). We pay for that politeness in wall time, visibly.
- 🔴 **`urllib3.Retry` does not come along.** It lives inside `requests`' transport.
  The async probe re-implements a small retry-on-5xx loop by hand — a reminder that
  switching HTTP stacks means re-earning your resilience features.

The `run_async` helper is **12.5**'s: bare `asyncio.run()` works in a plain script but
not in Jupyter, where an event loop is already running.

In [ ]:
import asyncio

import httpx


def run_async(coro):
    """Run a coroutine in a notebook or a plain script alike (12.5)."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)                  # no loop running: plain script
    with ThreadPoolExecutor(1) as pool:           # Jupyter: give it its own loop
        return pool.submit(asyncio.run, coro).result()


async def probe_service_async(client: httpx.AsyncClient, gate: asyncio.Semaphore,
                              name: str) -> ProbeResult:
    async with gate:                              # at most N requests in flight
        started = time.perf_counter()

        def done(outcome: Outcome, report: HealthReport | None = None,
                 detail: str = "") -> ProbeResult:
            return ProbeResult(name, outcome, report, detail,
                               (time.perf_counter() - started) * 1000)

        try:
            for attempt in range(3):              # retries by hand: Retry stayed behind
                response = await client.get(f"/{name}/health")
                if response.status_code < 500 or attempt == 2:
                    break
                await asyncio.sleep(0.1 * 2 ** attempt)
        except httpx.HTTPError as exc:
            return done(Outcome.UNREACHABLE, detail=type(exc).__name__)
        if response.status_code >= 400:
            return done(Outcome.UNREACHABLE, detail=f"HTTP {response.status_code}")
        try:
            payload = response.json()
        except ValueError:
            return done(Outcome.INVALID, detail="body is not JSON")
        try:
            report = HealthReport.model_validate(payload)
        except ValidationError as exc:
            return done(Outcome.INVALID, detail=f"{exc.error_count()} schema error(s)")
        outcome = Outcome.DEGRADED if report.status == "degraded" else Outcome.HEALTHY
        return done(outcome, report=report)


async def poll_fleet_async(limit: int = 4) -> list[ProbeResult]:
    gate = asyncio.Semaphore(limit)
    timeout = httpx.Timeout(2.0, connect=3.05)    # never optional, in any stack
    async with httpx.AsyncClient(base_url=BASE, timeout=timeout) as client:
        return list(await asyncio.gather(
            *(probe_service_async(client, gate, name) for name in FLEET)))


started = time.perf_counter()
async_results = run_async(poll_fleet_async(limit=4))
async_s = time.perf_counter() - started

same_verdicts = ({(r.service, r.outcome) for r in async_results}
                 == {(r.service, r.outcome) for r in threaded})
print(f"   {'sequential':<20}{sequential_s * 1000:>7.0f} ms")
print(f"   {'threads (12 workers)':<20}{threaded_s * 1000:>7.0f} ms")
print(f"   {'asyncio (gate of 4)':<20}{async_s * 1000:>7.0f} ms")
print(f"\n   all three strategies agree on all 12 verdicts: {same_verdicts}")

Read those three numbers as *this run's* numbers, not as physics. The shape that
repeats on this machine: both concurrent pollers come in ahead of the baseline, and the
thread pool comes in far ahead of the gated async poller — which only shaves roughly a
third off sequential here. Three reasons, and none of them is "asyncio is slow":

1. **The bound is doing its job.** At most 4 requests in flight versus the pool's 12 —
   politeness paid for in wall time, exactly as advertised (**18.3**).
2. **The hand-rolled retry backs off inside a semaphore slot.** While `search-legacy`
   sleeps between attempts, it is occupying one of only four lanes.
3. **`httpx` plus an event loop does more per request than `requests` in a thread** —
   and on loopback, where the waits are tiny, fixed per-request costs dominate
   (**18.5** measured the same effect and drew the same conclusion).

🔴 **Do not conclude a universal ranking from this loopback run** — the gap between the
two concurrent versions swings with the semaphore size, the retry timing, and the
server being a thread-per-connection toy. **18.5** measured this honestly and its
conclusion stands unchanged here:

> Threads are the right default for *tens* of calls — one line, works with `requests`.
> asyncio earns its complexity at *thousands* of concurrent connections, or when your
> framework is already async. **Measure your own workload; the bottleneck's shape
> decides.**

What this notebook adds to that conclusion: whichever poller you choose, **the verdicts
are identical** — transport strategy and trust policy are independent layers.

## Part 6 — The fleet report

The pollers return `list[ProbeResult]`; the report is one **pure function** over that
list — no HTTP, trivially testable (**15.3**). Rendered as an aligned table with format
specs (**1.3**): `<` pads names left, `>` right-aligns numbers.

In [ ]:
from collections import Counter


def render_fleet_report(results: list[ProbeResult]) -> str:
    """One consolidated report: what the on-call engineer actually reads."""
    order = list(FLEET)
    rows = sorted(results, key=lambda r: order.index(r.service))
    counts = Counter(r.outcome for r in rows)
    reachable = [r for r in rows if r.report is not None]
    worst = max(reachable, key=lambda r: r.report.latency_ms)
    versions = Counter(r.report.version for r in reachable)
    serving = counts[Outcome.HEALTHY] + counts[Outcome.DEGRADED]

    lines = [f"{'SERVICE':<15}{'OUTCOME':<13}{'LATENCY':>9}{'VERSION':>9}  DETAIL",
             "-" * 66]
    for r in rows:
        latency = f"{r.report.latency_ms:.0f} ms" if r.report else "-"
        version = r.report.version if r.report else "-"
        lines.append(f"{r.service:<15}{r.outcome.value:<13}{latency:>9}{version:>9}"
                     f"  {r.detail}")
    lines += [
        "-" * 66,
        f"{'fleet':<15}{serving}/{len(rows)} serving   "
        f"healthy={counts[Outcome.HEALTHY]} degraded={counts[Outcome.DEGRADED]} "
        f"unreachable={counts[Outcome.UNREACHABLE]} invalid={counts[Outcome.INVALID]}",
        f"{'worst latency':<15}{worst.report.latency_ms:.0f} ms self-reported,"
        f" by {worst.service}",
        f"{'version spread':<15}"
        + ", ".join(f"{v} x{n}" for v, n in versions.most_common()),
        f"{'verdict':<15}"
        + ("ALL SYSTEMS GO" if serving == len(rows) and not counts[Outcome.DEGRADED]
           else "NEEDS ATTENTION"),
    ]
    return "\n".join(lines)


print(render_fleet_report(threaded))

One glance answers the on-call question: nine of twelve serving, `shipping` degraded,
`search-legacy` down, two services whose payloads cannot be believed — and a version
spread showing the fleet is running three releases at once, which is how the `reports`
drift happened in the first place.

Worth pausing on what did *not* happen: the malformed body, the drifted schema and the
dead service all flowed **through** the pipeline as data. The boundary absorbed them
(**18.4**); the aggregation stayed a pure function; nothing crashed.

Note the two latencies in play: the table's `LATENCY` column is what each service
*claims* about itself, while `ProbeResult.elapsed_ms` is what we *measured*. A real
dashboard shows both — a service that self-reports 50 ms while your probes take 2 s is
its own kind of incident.

## Part 7 — Tests

The client follows **18.5**'s testing argument end to end:

- 🔴 **Do not mock `requests`** — that tests your beliefs about a library (**15.5**).
  `probe_service` *accepts its session*, so a unit test injects a `FakeSession` that
  answers `.get()` and records the call. Fast, no sockets.
- **One integration layer against a real local server** — a session-scoped pytest
  fixture (**15.4**) exercising real HTTP: sockets, headers, status codes.

The tests run as a real pytest project in a **temp directory** (nothing written next to
this notebook), via `subprocess` — the same module the fleet uses, packaged importably.

In [ ]:
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py194_"))


def write(rel: str, source: str) -> Path:
    path = WORK / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    return path


# The client layer as an importable module - identical logic to Part 2.
write("health_client.py", r"""
    # The aggregator's client layer: session, contract, probe.
    import enum
    import time
    from dataclasses import dataclass
    from typing import Annotated, Literal

    import requests
    from pydantic import BaseModel, Field, ValidationError
    from requests.adapters import HTTPAdapter
    from urllib3.util import Retry

    TIMEOUT: tuple[float, float] = (3.05, 2.0)


    class HealthReport(BaseModel):
        status: Literal["healthy", "degraded"]
        latency_ms: Annotated[float, Field(ge=0)]
        version: Annotated[str, Field(pattern=r"^\d+\.\d+\.\d+$")]


    class Outcome(enum.StrEnum):
        HEALTHY = "healthy"
        DEGRADED = "degraded"
        UNREACHABLE = "unreachable"
        INVALID = "invalid"


    @dataclass(frozen=True, slots=True)
    class ProbeResult:
        service: str
        outcome: Outcome
        report: HealthReport | None
        detail: str
        elapsed_ms: float


    def build_session() -> requests.Session:
        policy = Retry(total=2, backoff_factor=0.1, backoff_jitter=0.05,
                       status_forcelist=[502, 503, 504],
                       allowed_methods={"GET"}, raise_on_status=False)
        session = requests.Session()
        adapter = HTTPAdapter(max_retries=policy)
        session.mount("http://", adapter)
        session.mount("https://", adapter)
        return session


    def probe_service(session, base: str, name: str,
                      timeout: tuple[float, float] = TIMEOUT) -> ProbeResult:
        # Poll one /health endpoint. Never raises - failures become verdicts.
        started = time.perf_counter()

        def done(outcome, report=None, detail=""):
            return ProbeResult(name, outcome, report, detail,
                               (time.perf_counter() - started) * 1000)

        try:
            with session.get(f"{base}/{name}/health", timeout=timeout) as response:
                response.raise_for_status()
                payload = response.json()
        except requests.JSONDecodeError:
            return done(Outcome.INVALID, detail="body is not JSON")
        except requests.HTTPError as exc:
            return done(Outcome.UNREACHABLE, detail=f"HTTP {exc.response.status_code}")
        except requests.RequestException as exc:
            return done(Outcome.UNREACHABLE, detail=type(exc).__name__)
        try:
            report = HealthReport.model_validate(payload)
        except ValidationError as exc:
            return done(Outcome.INVALID, detail=f"{exc.error_count()} schema error(s)")
        outcome = Outcome.DEGRADED if report.status == "degraded" else Outcome.HEALTHY
        return done(outcome, report=report)
""")

write("test_unit.py", r"""
    import pytest
    import requests

    from health_client import HealthReport, Outcome, probe_service


    class FakeResponse:
        def __init__(self, status, payload=None, text=""):
            self.status_code = status
            self._payload = payload
            self.text = text
            self.closed = False

        def __enter__(self):              # the client uses the response as a
            return self                   # context manager, so the fake must too

        def __exit__(self, *exc_info):
            self.closed = True
            return False                  # never swallow the exception

        def json(self):
            if self._payload is None:
                raise requests.JSONDecodeError("Expecting value", self.text, 0)
            return self._payload

        def raise_for_status(self):
            if self.status_code >= 400:
                raise requests.HTTPError(f"{self.status_code}", response=self)


    class FakeSession:
        # A fake at OUR boundary (15.5): answers .get(), records the call.

        def __init__(self, responses):
            self._responses = list(responses)
            self.calls = []

        def get(self, url, timeout=None):
            self.calls.append((url, timeout))
            return self._responses.pop(0)


    def test_healthy_service_is_parsed_and_typed():
        session = FakeSession([FakeResponse(200, {"status": "healthy",
                                                  "latency_ms": "50",  # JSON strings happen
                                                  "version": "2.3.1"})])
        result = probe_service(session, "http://fleet.test", "gateway")
        assert result.outcome is Outcome.HEALTHY
        assert result.report == HealthReport(status="healthy", latency_ms=50.0,
                                             version="2.3.1")
        assert isinstance(result.report.latency_ms, float)   # coerced at the boundary (18.4)


    def test_degraded_status_is_its_own_outcome():
        session = FakeSession([FakeResponse(200, {"status": "degraded",
                                                  "latency_ms": 200,
                                                  "version": "2.2.9"})])
        result = probe_service(session, "http://fleet.test", "shipping")
        assert result.outcome is Outcome.DEGRADED


    def test_schema_drift_is_invalid_not_a_crash():
        session = FakeSession([FakeResponse(200, {"state": "up", "ver": 4})])
        result = probe_service(session, "http://fleet.test", "reports")
        assert result.outcome is Outcome.INVALID
        assert result.report is None
        assert "schema error" in result.detail


    def test_malformed_body_is_invalid():
        session = FakeSession([FakeResponse(200, payload=None, text="not json{{{")])
        result = probe_service(session, "http://fleet.test", "metrics")
        assert result.outcome is Outcome.INVALID
        assert result.detail == "body is not JSON"


    def test_server_error_is_unreachable():
        session = FakeSession([FakeResponse(503, {"error": "boom"})])
        result = probe_service(session, "http://fleet.test", "search-legacy")
        assert result.outcome is Outcome.UNREACHABLE
        assert result.detail == "HTTP 503"


    def test_response_is_closed_even_when_handling_raises():
        # The connection-leak regression test: an errored response must not
        # stay open until the garbage collector finds it.
        response = FakeResponse(503, {"error": "boom"})
        session = FakeSession([response])
        probe_service(session, "http://fleet.test", "search-legacy")
        assert response.closed is True


    def test_timeout_is_passed_through():
        # 18.5: the assertion a patched requests.get makes awkward.
        session = FakeSession([FakeResponse(200, {"status": "healthy",
                                                  "latency_ms": 1,
                                                  "version": "1.0.0"})])
        probe_service(session, "http://fleet.test", "auth", timeout=(1.0, 4.0))
        url, timeout = session.calls[0]
        assert url == "http://fleet.test/auth/health"
        assert timeout == (1.0, 4.0)
""")

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--no-header",
     "test_unit.py"],
    cwd=WORK, capture_output=True, text=True, encoding="utf-8", timeout=300)
print("$ pytest test_unit.py")
print("-" * 68)
print((result.stdout + result.stderr).strip())

In [ ]:
# The integration layer: a real (tiny) fleet server as a session-scoped fixture (15.4).
write("conftest.py", r"""
    import http.server
    import json
    import socket
    import threading

    import pytest


    class TinyFleet(http.server.BaseHTTPRequestHandler):
        protocol_version = "HTTP/1.1"

        def log_message(self, *args):
            pass

        def do_GET(self):
            if self.path == "/gateway/health":
                body = json.dumps({"status": "healthy", "latency_ms": 5,
                                   "version": "2.3.1"}).encode("utf-8")
                status = 200
            elif self.path == "/metrics/health":
                body, status = b'{"status": "healthy", "latency_ms": ', 200
            else:
                body, status = b'{"error": "no such service"}', 404
            self.send_response(status)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(body)))
            self.end_headers()
            self.wfile.write(body)


    class QuietServer(http.server.ThreadingHTTPServer):
        daemon_threads = True

        def handle_error(self, *args):
            pass


    @pytest.fixture(scope="session")
    def fleet_url():
        probe = socket.socket()
        probe.bind(("127.0.0.1", 0))
        port = probe.getsockname()[1]
        probe.close()
        server = QuietServer(("127.0.0.1", port), TinyFleet)
        threading.Thread(target=server.serve_forever, daemon=True).start()
        yield f"http://127.0.0.1:{port}"
        server.shutdown()
        server.server_close()


    @pytest.fixture
    def real_session():
        from health_client import build_session
        session = build_session()
        yield session
        session.close()
""")

write("test_integration.py", r"""
    from health_client import Outcome, probe_service


    def test_probe_over_real_http(fleet_url, real_session):
        result = probe_service(real_session, fleet_url, "gateway")
        assert result.outcome is Outcome.HEALTHY
        assert result.report.version == "2.3.1"
        assert result.elapsed_ms > 0


    def test_real_malformed_body_is_invalid(fleet_url, real_session):
        result = probe_service(real_session, fleet_url, "metrics")
        assert result.outcome is Outcome.INVALID
        assert result.detail == "body is not JSON"
""")

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--no-header",
     "test_unit.py", "test_integration.py"],
    cwd=WORK, capture_output=True, text=True, encoding="utf-8", timeout=300)
print("$ pytest test_unit.py test_integration.py")
print("-" * 68)
print((result.stdout + result.stderr).strip())

Nine tests, all offline: seven unit tests that never touch a socket — the timeout
pass-through check, one proving that **schema drift is a verdict, not a crash**, and
one pinning that the client closes every response it opens — plus two that exercise
the full stack over real HTTP, against a server that also serves deliberately broken
JSON. The fixture shuts its server down in its teardown half (**15.4**), just as the
next cell shuts down ours.

## Teardown

Everything this notebook started, stopped — and verified stopped, the way **12.2**
ends: by asking `threading.enumerate()` who is still alive.

In [ ]:
# ---- tidy up: close the client, stop the fleet, remove the scratch ----
import shutil

SESSION.close()                    # returns pooled connections, closes sockets
SERVER.shutdown()                  # stops serve_forever
SERVER.server_close()              # closes the listening socket
shutil.rmtree(WORK, ignore_errors=True)

deadline = time.perf_counter() + 5.0          # give handler threads a moment to unwind
while (any(t is not threading.main_thread() for t in threading.enumerate())
       and time.perf_counter() < deadline):
    time.sleep(0.05)

leftover = [t.name for t in threading.enumerate() if t is not threading.main_thread()]
print("fleet stopped, scratch removed:", not WORK.exists())
print("requests the fleet handled  :", sum(HITS.values()))
print("threads still alive         :", leftover or "none")
if leftover:
    print("  ^ in Jupyter these are the kernel's own service threads, not ours.")

---
## Common Mistakes & Pitfalls

1. 🔴 **A request without a timeout, under concurrency.** It does not just hang — it holds a worker forever, and a pool of twelve can be fully consumed by twelve hung probes (**18.5**).
2. 🔴 **Letting one bad service kill the poll.** `probe_service` never raises; every failure becomes a verdict. A dashboard that dies when a service lies is worse than the outage (**18.4**).
3. **Trusting `response.json()`.** It is `Any`, and `Any` spreads (**16.1**) — `metrics` proved that a `200` with a JSON `Content-Type` guarantees nothing.
4. **Retrying the wrong things.** `GET /health` is idempotent, so retrying is safe; the same policy on a `POST` without an idempotency key duplicates writes (**18.3**).
5. 🔴 **Mocking `requests` in the tests.** Inject a fake at *your* boundary instead — `probe_service` takes its session precisely so the tests can (**15.5**, **18.5**).
6. **Unbounded concurrency.** Twelve at once is fine against your own fleet; 500 at once against someone else's API is a self-inflicted incident. `max_workers` and `Semaphore` are the controls (**12.5**, **18.3**).
7. 🔴 **Concluding a universal threads-vs-asyncio ranking from one loopback run.** The ordering here moves with the semaphore size, retry timing and a toy server — measure your own workload (**18.5**, **17.5**).
8. **Believing self-reported `latency_ms`.** Measure `elapsed_ms` yourself; a service that claims 50 ms while probes take 2 s is lying in a way only your own clock catches.
9. 🔴 **Keeping `Response` objects alive.** A retained response — a notebook global, a cache entry, a stored exception — holds a socket file descriptor open even after `session.close()`. Extract the data and let the response go; an early draft of this notebook kept three peek responses as globals and paid for it with a leftover server thread at teardown.
10. **Forgetting the teardown.** An unshut server leaks threads and its listening socket; an unclosed session leaks connections. `threading.enumerate()` is the audit (**12.2**).
11. **Assuming the retry adapter follows you to `httpx`.** `urllib3.Retry` is `requests`' transport; switch stacks and you re-earn resilience by hand, as the async probe did.

## Best Practices

- One `Session` per poll cycle, retries mounted on the adapter, timeout on every request — the three-line foundation before any concurrency.
- Validate at the boundary with one `pydantic` model; hand the rest of the program frozen, typed results (**18.4**).
- Make the probe **total**: every possible failure maps to a structured verdict, and the taxonomy (`UNREACHABLE` vs `INVALID`) tells on-call *where* to look.
- Collect with `as_completed` so consumers can act on early results (**12.4**).
- Bound concurrency deliberately, and keep the bound in one obvious place (**18.3**).
- Keep aggregation a pure function over `list[ProbeResult]` — transport and reporting stay independently testable (**15.3**).
- Unit-test with a fake session, integration-test against a real local server, and keep both offline (**15.4**, **15.5**).
- Record both latencies: what the service claims and what you measured.
- Shut down what you start, and let `threading.enumerate()` prove it (**12.2**).

## Extension exercises

Try these before calling the capstone done.

1. **A TTL cache.** Store each `ProbeResult` with a `time.monotonic()` stamp and serve cached verdicts for 10 s — a dashboard refreshing every second should not re-poll the fleet every second.
2. 🔴 **Become a service.** Expose the aggregate as your own `GET /status` endpoint on a `ThreadingHTTPServer`, returning the report as JSON with an appropriate status code (`200` all-serving, `503` otherwise). Congratulations: someone will now aggregate *you*.
3. **A circuit breaker per service (18.3).** After 3 consecutive `UNREACHABLE` verdicts, skip `search-legacy` for 30 s and mark it `unreachable (circuit open)` — stop paying the retry tax every poll.
4. **Scheduling.** Run the poll every 30 s. Start with `while True: poll(); time.sleep(30)`, notice the drift (each cycle slips by the poll's duration), fix it by sleeping until the *next deadline* — then look at `sched`, APScheduler or cron for the real thing.
5. **Flap detection.** Keep the last N reports per service and flag services that changed verdict more than twice in the window — a flapping service is a different incident than a down one.
6. **Version-drift alarm.** Fail the fleet verdict when more than two versions are serving at once; the `reports` schema drift began as exactly that.
7. **`asyncio.TaskGroup` (12.5).** Rewrite `poll_fleet_async` with a `TaskGroup` (3.11+). What happens to the other probes when one task raises — and is that what a dashboard wants?
8. **A total-poll budget.** Give the whole poll a 1-second deadline: pass each probe the time remaining rather than a fixed timeout, and mark services that missed the window as `unreachable (budget)`.
9. **Interview question:** design the health poll for 5,000 hosts with a 100 req/s ceiling and a 2% failure rate. Which poller, what bound, what retry policy — and how long does one sweep take?

## What you built

A fleet poller that survives everything twelve misbehaving services could throw at it:
`Session` + adapter retries + timeouts as the transport (**18.3**, **18.5**), `pydantic`
as the trust boundary (**18.4**), threads and asyncio as interchangeable concurrency
layers (**12.4**, **12.5**), a pure aggregation function rendering one report (**1.3**),
and a test suite that needs no network and mocks nothing it does not own (**15.5**).

**The one-sentence version:** *overlap the waiting, bound the pressure, validate at the
boundary, and turn every failure into data — then the report is just a fold.*

## Related

- **18.5 Testing API Clients and Concurrency** — this capstone is that notebook, grown up
- **18.4 Validating What You Receive** — the trust boundary the probe enforces
- **18.3 Pagination, Rate Limits and Retries** — the adapter policy and the politeness bound
- **12.4 concurrent.futures / 12.5 asyncio** — the two concurrency layers, compared honestly
- **15.4 Fixtures / 15.5 Test Doubles** — the session-scoped server and the fake at the boundary
- **19.x** — the other capstones, each tying a different folder-span together